# 🧬 Project 5 — Semantic Correspondence
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup
Eseguire per configurare Drive e caricare le funzioni Helper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

In [ ]:
# ── Setup Logica Demo (Stage 4) ─────────────────────────────────
# Questo blocco carica le funzioni necessarie per le demo interattive.
import torch, os, gradio as gr
from PIL import Image, ImageDraw
import torchvision.transforms as T
from models.extractor import DINOv2Extractor
from models.lora import apply_lora_to_dinov2
from models.correspondence import SemanticCorrespondenceModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def launch_stage_demo(title, ckpt_name=None):
    # Caricamento Backbone
    backbone = DINOv2Extractor(model_name='dinov2_vitb14', freeze=True)
    if ckpt_name:
        path = f'/content/drive/MyDrive/AML/Checkpoints/{ckpt_name}/best.pth'
        if not os.path.exists(path):
            print(f"⚠️ Checkpoint '{ckpt_name}' non trovato in Drive. Uso la Baseline.")
            model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
        else:
            ckpt = torch.load(path, map_location=device)
            backbone.model = apply_lora_to_dinov2(backbone.model, rank=ckpt['args'].get('lora_rank', 16))
            model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
    else:
        model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
    
    model.eval()

    def predict(src_img, trg_img, x, y):
        s_t = transform(src_img).unsqueeze(0).to(device)
        t_t = transform(trg_img).unsqueeze(0).to(device)
        scale = (224 / src_img.width, 224 / src_img.height)
        src_kp = torch.tensor([[[x * scale[0], y * scale[1]]]], device=device).float()
        out = model(s_t, t_t, src_kps=src_kp)
        pkp = out['pred_kps'][0,0].cpu().numpy()
        return pkp[0] * (trg_img.width/224), pkp[1] * (trg_img.height/224)

    def gradio_fn(src_img, trg_img, evt: gr.SelectData):
        tx, ty = predict(src_img, trg_img, evt.index[0], evt.index[1])
        s_out, t_out = src_img.copy(), trg_img.copy()
        r = 6
        ImageDraw.Draw(s_out).ellipse([evt.index[0]-r, evt.index[1]-r, evt.index[0]+r, evt.index[1]+r], fill='red', outline='white')
        ImageDraw.Draw(t_out).ellipse([tx-r, ty-r, tx+r, ty+r], fill='green', outline='white')
        return s_out, t_out

    with gr.Blocks() as demo:
        gr.Markdown(f"### {title}")
        with gr.Row():
            src_in = gr.Image(type='pil', label='Source (Clicca)')
            trg_in = gr.Image(type='pil', label='Target')
        with gr.Row():
            src_res = gr.Image(type='pil', label='Input Click')
            trg_res = gr.Image(type='pil', label='Prediction')
        src_in.select(gradio_fn, [src_in, trg_in], [src_res, trg_res])
    demo.launch(share=True, inline=False)

## 🔍 1. Stage 1: Backbone Analysis
Analisi zero-shot della Baseline.

In [ ]:
# 1.1 PCK Baseline
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --results_file /content/drive/MyDrive/AML/Results/baseline.txt

In [ ]:
# 🎮 Demo Stage 1
launch_stage_demo('DINOv2 Baseline')

## 🚀 2. Stage 2: Fine-Tuning Efficiente (LoRA)
Addestramento modulare.

In [ ]:
# 2.1 Solo LoRA
!python train.py --epochs 5 --curriculum_epochs 0 --num_workers 4 --output_dir ./checkpoints/lora_only --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_only

In [ ]:
# 🎮 Demo Stage 2.1
launch_stage_demo('LoRA Only', ckpt_name='lora_only')

In [ ]:
# 2.2 LoRA + Curriculum
!python train.py --epochs 5 --curriculum_epochs 2 --num_workers 4 --output_dir ./checkpoints/lora_curriculum --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_curriculum

## 🎯 3. Stage 3: Raffinamento Adattivo
Efficacia dell'Adaptive Windowing.

In [ ]:
CKPT = '/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/best.pth'
!python evaluate.py --checkpoint {CKPT} --results_file /content/drive/MyDrive/AML/Results/s3_with.txt
!python evaluate.py --checkpoint {CKPT} --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/s3_without.txt

## 🌟 4. Stage 4: Valutazione ed Estensioni
Demo del modello finale.

In [ ]:
# 🎮 Demo Stage 4 (Miglior Modello)
launch_stage_demo('Modello Finale: LoRA + Curriculum', ckpt_name='lora_curriculum')